# V1 - Learning the Basics

In [ ]:
!pip install -q google-generativeai
import google.generativeai as genai
from google.colab import userdata
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))  # store key in Colab's Secrets tab (key icon, left sidebar)

First we're going to set up our environment variable to make use of our Gemini API Key

In [ ]:
job_description = input("Paste job description: ")  # or just a triple-quoted string cell for pasting long text
resume = input("Paste resume: ")
github_username = input("GitHub username (optional): ")

Paste job description: Job description At NiCE, we don’t limit our challenges. We challenge our limits. Always. We’re ambitious. We’re game changers. And we play to win. We set the highest standards and execute beyond them. And if you’re like us, we can offer you the ultimate career opportunity that will light a fire within you.  So, what’s the role all about?  This is not just another full stack role—this is a chance to help modernize a core NICE platform at scale. You’ll join a high-visibility engineering team driving the transformation of a legacy UI (ASPX/.NET) into modern Angular/React front ends powered by microservices architecture.  You’ll be part of the Novus team, a fast-moving, highly collaborative group at the center of NICE’s innovation efforts—working closely with engineers, architects, and cross-functional partners.  Even more exciting: this team is actively leveraging AI tools to accelerate development, automate migrations, and rethink how software is built—giving you h

Some simple python to prompt the user for a few items to test out our system

In [ ]:
import requests
repos = []
if github_username:
    r = requests.get(f"https://api.github.com/users/{github_username}/repos", timeout=10)
    data = r.json()
    if r.status_code == 200 and isinstance(data, list):
        repos = [{"name": x["name"], "desc": x.get("description"), "lang": x.get("language")} for x in data]
    else:
        # A bad username returns a dict like {"message": "Not Found"}, not a list —
        # iterating that would crash, so degrade gracefully instead.
        print(f"⚠️ Could not fetch repos for '{github_username}' (HTTP {r.status_code}). Continuing without GitHub data.")

Optional Call to Github API to fetch repos that may or may not be a fit

In [8]:
for m in genai.list_models():
  print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [9]:
model = genai.GenerativeModel("gemini-flash-latest")
prompt = f"""
You are a resume tailoring assistant...
JOB DESCRIPTION: {job_description}
RESUME: {resume}
GITHUB PROJECTS: {repos}

Return two sections:
1. TAILORED RESUME (markdown)
2. WHAT CHANGED AND WHY (bulleted)
"""
response = model.generate_content(prompt)
print(response.text)

# Jordan Alvarez
jordan.alvarez@email.com | (555) 123-4567 | github.com/jalvarez-dev | Atlanta, GA

## Summary
Dynamic and collaborative Full Stack Engineer with a proven track record of modernizing legacy systems, building scalable web applications, and driving development efficiency. Adept at transforming monolithic architectures into high-performance, modular APIs and responsive front-end applications (React/Angular). Actively leverages next-generation AI-assisted engineering tools to accelerate migrations, automate workflows, and boost engineering productivity. Committed to end-to-end quality, cross-functional collaboration, and delivering exceptional customer experiences.

## Experience

**Software Engineer** — Bright Path Logistics | Jun 2023 – Present
* **Legacy Modernization:** Spearheaded the modernization of a legacy monolithic shipment-tracking system, rebuilding it into a high-performance full-stack dashboard utilizing React and Python modular APIs, improving UI responsiven

# V1.1 - Improving the Foundation

*(Interim step between V1 and V2 — summarized here rather than shown as separate cells, since it reuses V1's single-prompt approach.)*

V1.1 keeps V1's one-shot prompt but hardens the input/output that a live workshop kept tripping over:

- **Job description by URL** — fetch and strip the page with BeautifulSoup, not just pasted text.
- **Resume by file upload** — accept a `.md`/`.txt` upload, not just pasted text.
- **Graceful GitHub failures** — wrap the repo fetch so a bad username or rate limit degrades to "no repo evidence" instead of crashing.

The next notebook (**V2**) rebuilds this foundation with production patterns: real RAG (chunking, embeddings, retrieval, evaluation), a multi-tool agent, and AI safety guardrails.